# Overview

Reproduction of Diamantini et al. (2024) on Rossmann Store Sales (non-normalized `Sales`) with Transformer, LSTM, MLP, RNN, and CNN.

This notebook supports both **local / JupyterHub** and **Google Colab** runtimes. Toggle `USE_GOOGLE_COLAB` in the **Runtime Configuration** section (right after Model Architectures).


# Model Architectures

Diamantini et al. (2024) compare several neural-network architectures for KPI forecasting. Because full hyperparameter details are not fully documented in the paper, this notebook uses explicit equivalent baselines:

- **Transformer** — stacked encoder blocks with multi-head attention.
- **LSTM** — stacked recurrent layers.
- **MLP** — dense feed-forward network.
- **RNN** — stacked SimpleRNN layers.
- **CNN** — stacked Conv1D blocks.

For stability on raw `Sales`, models use output-layer bias initialized to `mean(y_train)`, per-model learning rates, gradient clipping, and `TerminateOnNaN`.


# Runtime Configuration

Set `USE_GOOGLE_COLAB` below to choose the runtime:

- `False` (default) — local / JupyterHub: load Rossmann from `data/raw/rossmann` under the project root.
- `True` — Google Colab: mount Drive and load Rossmann from the Shared Drive path used in the Colab exploration notebook.

GPU usage is detected dynamically in either mode (GPU when available, otherwise CPU).


In [ ]:
# ============================================================
# Runtime toggle — set this before running setup cells below
# ============================================================
USE_GOOGLE_COLAB = True  # True = Google Colab + Shared Drive; False = local / JupyterHub

# Shared Drive path used when USE_GOOGLE_COLAB=True (same as diamantini_2024_rossmann-old)
COLAB_ROSSMANN_DIR = (
    '/content/drive/Shareddrives/Riset S3/Forecasting/'
    'dataset-drug-demand-prediction/rossmann'
)

print(f'USE_GOOGLE_COLAB = {USE_GOOGLE_COLAB}')
if USE_GOOGLE_COLAB:
    print(f'Colab Rossmann dir : {COLAB_ROSSMANN_DIR}')
else:
    print('Dataset will be resolved from project root: data/raw/rossmann')


USE_GOOGLE_COLAB = True
Colab Rossmann dir : /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/rossmann


# Reference-Based Exploration

Exploration follows the Diamantini et al. (2024) Rossmann non-normalized forecasting setup. Preprocessing loads and merges `train.csv` + `store.csv`, engineers promo-related features, applies a chronological 80/20 split, and scales input features only (target remains raw). Modeling trains Transformer, LSTM, MLP, RNN, and CNN, then compares reproduction metrics against verified paper reference values.


## Preprocessing

Runtime setup, paper reference validation, dataset loading/validation, exploratory data analysis, and modeling data preparation.

Dataset location follows `USE_GOOGLE_COLAB` from **Runtime Configuration**:
- Colab: Shared Drive Rossmann folder (after `drive.mount`)
- Local / JupyterHub: `data/raw/rossmann` under the project root

Compute device detection uses **GPU when available**, otherwise falls back to **CPU** with explicit log messages.


In [2]:
import sys
import os
import random
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


def resolve_project_root() -> Path:
    """Resolve repo root when running from notebooks/ or similar."""
    root = Path(os.getcwd()).resolve()
    if root.name in {'explore', 'notebooks', 'Notebooks-to-transfers'}:
        root = root.parent
        if root.name == 'notebooks':
            root = root.parent
    return root


def configure_colab_runtime(use_google_colab: bool, colab_rossmann_dir: str):
    """Mount Drive and return Colab Rossmann dir when enabled; else None."""
    if not use_google_colab:
        return None

    try:
        from google.colab import drive
    except ImportError as exc:
        raise ImportError(
            'USE_GOOGLE_COLAB=True but google.colab is unavailable. '
            'Run this notebook in Google Colab, or set USE_GOOGLE_COLAB=False.'
        ) from exc

    drive.mount('/content/drive')
    data_dir = Path(colab_rossmann_dir)
    print('[INFO] Google Colab mode enabled.')
    print(f'[INFO] Mounted Drive; Rossmann dir -> {data_dir}')
    return data_dir


def configure_compute_device(project_root: Path) -> dict:
    """Detect GPU/CPU dynamically and print a clear training environment summary."""
    physical_gpus = tf.config.list_physical_devices('GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')

    for gpu in physical_gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as exc:
            print(f'[WARN] Could not enable GPU memory growth: {exc}')

    gpu_available = len(logical_gpus) > 0
    if gpu_available:
        compute_device = 'GPU'
        device_name = physical_gpus[0].name
        try:
            details = tf.config.experimental.get_device_details(physical_gpus[0])
            device_name = details.get('device_name', device_name)
        except Exception:
            pass
        training_strategy = (
            tf.distribute.MirroredStrategy()
            if len(logical_gpus) > 1
            else tf.distribute.get_strategy()
        )
        device_message = '[INFO] GPU detected — deep learning training will use GPU acceleration.'
    else:
        compute_device = 'CPU'
        logical_cpus = tf.config.list_logical_devices('CPU')
        device_name = logical_cpus[0].name if logical_cpus else '/CPU:0'
        training_strategy = tf.distribute.get_strategy()
        device_message = '[WARN] No GPU detected — deep learning training will run on CPU (slower).'

    # Keep float32: raw Sales target + MSE can overflow with mixed_float16.
    keras.mixed_precision.set_global_policy('float32')

    print('=' * 60)
    print('Compute environment')
    print('=' * 60)
    print(f'USE_GOOGLE_COLAB  : {USE_GOOGLE_COLAB}')
    print(f'Project root       : {project_root}')
    print(f'TensorFlow version : {tf.__version__}')
    print(device_message)
    print(f'Compute device     : {compute_device}')
    print(f'Device name        : {device_name}')
    print(f'Physical GPUs      : {physical_gpus}')
    print(f'Logical GPUs       : {logical_gpus}')
    print(f'Training strategy  : {type(training_strategy).__name__}')
    print(f'Mixed precision    : {keras.mixed_precision.global_policy()}')
    print('=' * 60)

    return {
        'gpu_available': gpu_available,
        'compute_device': compute_device,
        'device_name': device_name,
        'training_strategy': training_strategy,
        'physical_gpus': physical_gpus,
        'logical_gpus': logical_gpus,
    }


if 'USE_GOOGLE_COLAB' not in globals():
    raise NameError(
        'USE_GOOGLE_COLAB is not defined. Run the Runtime Configuration cell first.'
    )

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

COLAB_DATA_DIR = configure_colab_runtime(USE_GOOGLE_COLAB, COLAB_ROSSMANN_DIR)

compute = configure_compute_device(PROJECT_ROOT)

GPU_AVAILABLE = compute['gpu_available']
COMPUTE_DEVICE = compute['compute_device']
DEVICE_NAME = compute['device_name']
TRAINING_STRATEGY = compute['training_strategy']
GPU_DEVICES = compute['physical_gpus']
LOGICAL_GPUS = compute['logical_gpus']

print('All imports successful.')


Mounted at /content/drive
[INFO] Google Colab mode enabled.
[INFO] Mounted Drive; Rossmann dir -> /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/rossmann
[WARN] Could not enable GPU memory growth: Physical devices cannot be modified after being initialized
Compute environment
USE_GOOGLE_COLAB  : True
Project root       : /content
TensorFlow version : 2.20.0
[INFO] GPU detected — deep learning training will use GPU acceleration.
Compute device     : GPU
Device name        : Tesla T4
Physical GPUs      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPUs       : [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
Training strategy  : _DefaultDistributionStrategy
Mixed precision    : <DTypePolicy "float32">
All imports successful.


In [3]:
# Rossmann dataset location (follows USE_GOOGLE_COLAB)
if USE_GOOGLE_COLAB:
    DATA_DIR = COLAB_DATA_DIR if COLAB_DATA_DIR is not None else Path(COLAB_ROSSMANN_DIR)
else:
    DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'rossmann'

TRAIN_PATH = DATA_DIR / 'train.csv'
STORE_PATH = DATA_DIR / 'store.csv'

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'train.csv not found: {TRAIN_PATH}')
if not STORE_PATH.exists():
    raise FileNotFoundError(f'store.csv not found: {STORE_PATH}')

print(f'Runtime mode   : {"Google Colab" if USE_GOOGLE_COLAB else "Local / JupyterHub"}')
print(f'Data directory : {DATA_DIR}')
print(f'Train path     : {TRAIN_PATH}')
print(f'Store path     : {STORE_PATH}')


Runtime mode   : Google Colab
Data directory : /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/rossmann
Train path     : /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/rossmann/train.csv
Store path     : /content/drive/Shareddrives/Riset S3/Forecasting/dataset-drug-demand-prediction/rossmann/store.csv


### Paper Reference Validation

The initial task extraction showed some RMSE/MAE values misaligned with the original paper table. This notebook keeps two sources:

- `SOURCE_EXTRACTED_REFERENCE` — values from the initial extraction/task description.
- `PAPER_REFERENCE` — verified values used for final comparison.

Note: Table 1 in the paper uses column order **MAE, MSE, RMSE, R Squared**. With this order, some initially extracted RMSE values actually belong to other models/metrics.


In [6]:
SOURCE_EXTRACTED_REFERENCE = {
    'Transformer': {'rmse': 1283.77, 'mse': 1475395.99, 'r2': 0.96, 'mae': 800.01},
    'LSTM': {'rmse': 1394.37, 'mse': 1646816.85, 'r2': 0.91, 'mae': 821.29},
    'MLP': {'rmse': 1932.60, 'mse': 1750258.53, 'r2': 0.89, 'mae': 1255.74},
    'RNN': {'rmse': np.nan, 'mse': 1945628.38, 'r2': 0.88, 'mae': 458.91},
    'CNN': {'rmse': np.nan, 'mse': 3734871.20, 'r2': 0.74, 'mae': np.nan},
}

PAPER_REFERENCE = {
    'MLP': {'mae': 835.54, 'mse': 1750258.53, 'rmse': 1322.57, 'r2': 0.89},
    'LSTM': {'mae': 800.01, 'mse': 1646816.85, 'rmse': 1283.77, 'r2': 0.91},
    'CNN': {'mae': 1255.74, 'mse': 3734871.20, 'rmse': 1932.60, 'r2': 0.74},
    'RNN': {'mae': 821.29, 'mse': 1945628.38, 'rmse': 1394.37, 'r2': 0.88},
    'Transformer': {'mae': 760.48, 'mse': 1475395.99, 'rmse': 1214.00, 'r2': 0.96},
}

source_df = pd.DataFrame(SOURCE_EXTRACTED_REFERENCE).T.add_prefix('source_')
paper_df_reference = pd.DataFrame(PAPER_REFERENCE).T.add_prefix('paper_')
paper_verification = source_df.join(paper_df_reference, how='outer')

for metric in ['rmse', 'mse', 'r2', 'mae']:
    paper_verification[f'{metric}_mismatch'] = ~np.isclose(
        paper_verification[f'source_{metric}'].astype(float),
        paper_verification[f'paper_{metric}'].astype(float),
        equal_nan=True,
    )

paper_verification


,source_rmse,source_mse,source_r2,source_mae,paper_mae,paper_mse,paper_rmse,paper_r2,rmse_mismatch,mse_mismatch,r2_mismatch,mae_mismatch
CNN,NaN,3734871.20,0.74,NaN,1255.74,3734871.20,1932.60,0.74,True,False,False,True
LSTM,1394.37,1646816.85,0.91,821.29,800.01,1646816.85,1283.77,0.91,True,False,False,True
MLP,1932.60,1750258.53,0.89,1255.74,835.54,1750258.53,1322.57,0.89,True,False,False,True
RNN,NaN,1945628.38,0.88,458.91,821.29,1945628.38,1394.37,0.88,True,False,False,True
Transformer,1283.77,1475395.99,0.96,800.01,760.48,1475395.99,1214.00,0.96,True,False,False,True


The table above separates initial extracted values from verified paper reference values. Mismatch columns highlight metrics that do not align with the PDF extraction, so final evaluation uses explicit `paper_*` values.


### Dataset Validation

Load `train.csv` and `store.csv`, merge them, and run initial checks: dataset size, date range, store count, missing values, duplicates, zero-sales rows, and closed-store rows.


In [4]:
def preprocess_rossmann(train_path: Path, store_path: Path) -> pd.DataFrame:
    train = pd.read_csv(train_path, low_memory=False)
    store = pd.read_csv(store_path, low_memory=False)
    data = train.merge(store, how='left', on='Store')
    data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
    data = data.sort_values(['Date', 'Store']).reset_index(drop=True)

    data['MonthStr'] = data['Date'].dt.strftime('%b')

    def is_promo2_active(row):
        if row['Promo2'] == 1 and isinstance(row['PromoInterval'], str):
            return int(row['MonthStr'] in row['PromoInterval'].split(','))
        return 0

    data['IsPromo2Active'] = data.apply(is_promo2_active, axis=1)
    promo2_weeks = ((data['Date'].dt.year - data['Promo2SinceYear']) * 52 +
                    (data['Date'].dt.isocalendar().week.astype(float) - data['Promo2SinceWeek']))
    data['Promo2DurationWeeks'] = np.where(data['Promo2'] == 1, promo2_weeks, 0)
    data['Promo2DurationWeeks'] = data['Promo2DurationWeeks'].clip(lower=0).fillna(0)

    fill_zero_cols = ['CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']
    data[fill_zero_cols] = data[fill_zero_cols].fillna(0)

    data = pd.get_dummies(
        data,
        columns=['StoreType', 'Assortment', 'StateHoliday'],
        drop_first=True,
    )
    bool_cols = data.select_dtypes(include='bool').columns
    data[bool_cols] = data[bool_cols].astype(int)
    data = data.drop(columns=['PromoInterval', 'MonthStr'], errors='ignore')
    data = data.dropna(subset=['Date', 'Sales'])
    return data


raw_df = preprocess_rossmann(TRAIN_PATH, STORE_PATH)
print(raw_df.shape)
print(raw_df['Date'].min(), raw_df['Date'].max())

(1017209, 24)
2013-01-01 00:00:00 2015-07-31 00:00:00


Merged dataset shape and date range confirm successful loading from the project data directory. This merged dataset is the base for temporal splitting, while `Sales` remains on the original scale.


### Modeling Data Preparation

Configuration:
- `OPEN_ONLY = False` — closed stores are retained.
- `POSITIVE_SALES_ONLY = False` — `Sales = 0` rows are retained.
- `MAX_ROWS = None` — full dataset is used.
- Split: time-ordered 80/20.
- Target `Sales` is not normalized; scaling is applied to input features only.


In [5]:
POSITIVE_SALES_ONLY = False
MAX_ROWS = None
TEST_SIZE = 0.20

df = raw_df.copy()
df = df[df['Open'] == 1].copy()

if POSITIVE_SALES_ONLY:
    df = df[df['Sales'] > 0].copy()
if MAX_ROWS is not None:
    df = df.tail(MAX_ROWS).copy()

n_train = int(len(df) * (1 - TEST_SIZE))
train_df = df.iloc[:n_train].copy()
test_df = df.iloc[n_train:].copy()

target_col = 'Sales'
feature_cols = [c for c in train_df.columns if c not in [target_col, 'Date']]

X_train_raw = train_df[feature_cols].to_numpy(dtype=np.float32)
y_train = train_df[target_col].to_numpy(dtype=np.float32)
X_test_raw = test_df[feature_cols].to_numpy(dtype=np.float32)
y_test = test_df[target_col].to_numpy(dtype=np.float32)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

# Safety for neural training: sklearn/TensorFlow must not receive NaN/Inf.
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_test = np.nan_to_num(y_test, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

assert np.isfinite(X_train).all(), 'X_train masih mengandung NaN/Inf'
assert np.isfinite(X_test).all(), 'X_test masih mengandung NaN/Inf'
assert np.isfinite(y_train).all(), 'y_train masih mengandung NaN/Inf'
assert np.isfinite(y_test).all(), 'y_test masih mengandung NaN/Inf'

print('rows:', len(df))
print('train:', len(train_df), 'test:', len(test_df), 'features:', len(feature_cols))
print('target normalized:', False)
print('X finite:', np.isfinite(X_train).all(), np.isfinite(X_test).all())
print('y range:', float(y_train.min()), float(y_train.max()))
pd.Series(y_train).describe()

rows: 844392
train: 675513 test: 168879 features: 22
target normalized: False
X finite: True True
y range: 0.0 38037.0


,0
count,675513.000000
mean,6914.669922
std,3112.561768
min,0.000000
25%,4812.000000
50%,6323.000000
75%,8316.000000
max,38037.000000


Train/test size, feature count, and finite checks for `X`/`y` confirm the data is ready for neural-network training. `y_train` and `y_test` remain non-normalized so RMSE, MSE, R², and MAE are directly comparable to the paper.


## Modeling

Train Transformer, LSTM, MLP, RNN, and CNN on the reference preprocessing pipeline. Each model is evaluated on the held-out test set using RMSE, MSE, R², and MAE.


### Model Implementation

Shared building blocks and metric helpers for all neural-network baselines.


In [6]:
def dense_block(x, units: int, dropout: float):
    x = layers.Dense(units, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    return x


def transformer_encoder_block(x, d_model: int, num_heads: int, ffn_units: int, dropout: float):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, attn]))
    ffn = layers.Dense(ffn_units, activation='relu')(x)
    ffn = layers.Dropout(dropout)(ffn)
    ffn = layers.Dense(d_model)(ffn)
    return layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, ffn]))


def output_regression_layer(output_bias=None):
    if output_bias is None:
        return layers.Dense(1)
    return layers.Dense(1, bias_initializer=keras.initializers.Constant(output_bias))


def build_model(model_name: str, n_features: int, learning_rate=1e-4, dropout=0.10, output_bias=None):
    if model_name == 'MLP':
        inputs = keras.Input(shape=(n_features,))
        x = dense_block(inputs, 1024, dropout)
        x = dense_block(x, 768, dropout)
        x = dense_block(x, 512, dropout)
        x = dense_block(x, 256, dropout)
        x = dense_block(x, 128, dropout)
        x = dense_block(x, 64, dropout)
        outputs = output_regression_layer(output_bias)(x)
    else:
        inputs = keras.Input(shape=(1, n_features))
        if model_name == 'LSTM':
            x = layers.LSTM(256, return_sequences=True, dropout=dropout)(inputs)
            x = layers.LSTM(128, return_sequences=True, dropout=dropout)(x)
            x = layers.LSTM(64, dropout=dropout)(x)
        elif model_name == 'RNN':
            x = layers.SimpleRNN(256, return_sequences=True, dropout=dropout)(inputs)
            x = layers.SimpleRNN(128, return_sequences=True, dropout=dropout)(x)
            x = layers.SimpleRNN(64, dropout=dropout)(x)
        elif model_name == 'CNN':
            x = layers.Conv1D(256, kernel_size=1, activation='relu')(inputs)
            x = layers.BatchNormalization()(x)
            x = layers.Conv1D(256, kernel_size=1, activation='relu')(x)
            x = layers.BatchNormalization()(x)
            x = layers.Conv1D(128, kernel_size=1, activation='relu')(x)
            x = layers.BatchNormalization()(x)
            x = layers.Conv1D(64, kernel_size=1, activation='relu')(x)
            x = layers.BatchNormalization()(x)
            x = layers.Flatten()(x)
        elif model_name == 'Transformer':
            d_model = 256
            num_heads = 8
            x = layers.Dense(d_model)(inputs)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=512, dropout=dropout)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=512, dropout=dropout)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=256, dropout=dropout)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=256, dropout=dropout)
            x = layers.GlobalAveragePooling1D()(x)
        else:
            raise ValueError(model_name)

        x = dense_block(x, 512, dropout)
        x = dense_block(x, 256, dropout)
        x = dense_block(x, 128, dropout)
        x = dense_block(x, 64, dropout)
        outputs = output_regression_layer(output_bias)(x)

    model = keras.Model(inputs, outputs, name=f'diamantini_rossmann_{model_name.lower()}')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=0.5),
        loss='mse',
        metrics=['mae'],
    )
    return model


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'rmse': float(np.sqrt(mse)),
        'mse': float(mse),
        'r2': float(r2_score(y_true, y_pred)),
        'mae': float(mean_absolute_error(y_true, y_pred)),
    }

#### Training

Run all models sequentially. TensorFlow automatically uses **GPU when available** and falls back to **CPU** otherwise. Batch size is auto-selected per device (`1024` on GPU, `256` on CPU). Results are collected in `repro_df`.


In [11]:
MODELS_TO_RUN = ['Transformer', 'LSTM', 'MLP', 'RNN', 'CNN']
EPOCHS = 100
BATCH_SIZE = 1024 if GPU_AVAILABLE else 256
VALIDATION_SPLIT = 0.10
PATIENCE = 12
OUTPUT_BIAS = float(np.mean(y_train))
MODEL_LEARNING_RATES = {
    'Transformer': 1e-4,
    'LSTM': 3e-4,
    'MLP': 5e-4,
    'RNN': 2e-4,
    'CNN': 5e-4,
}

X_train_seq = X_train[:, np.newaxis, :]
X_test_seq = X_test[:, np.newaxis, :]

print('=' * 60)
print('Training configuration')
print('=' * 60)
if GPU_AVAILABLE:
    print(f'[INFO] Using GPU for deep learning training ({DEVICE_NAME}).')
else:
    print('[WARN] GPU not available — falling back to CPU training.')
    print('[WARN] Expect significantly longer runtime on the full Rossmann dataset.')
print(f'Compute device : {COMPUTE_DEVICE}')
print(f'Device name    : {DEVICE_NAME}')
print(f'Batch size     : {BATCH_SIZE} (auto-selected for {COMPUTE_DEVICE})')
print(f'Output bias    : {OUTPUT_BIAS:.4f}')
print(f'Models to run  : {MODELS_TO_RUN}')
print('=' * 60)

repro_rows = []
histories = {}

for model_name in MODELS_TO_RUN:
    print(f'\n=== {model_name} ===')
    print(f'Compute device : {COMPUTE_DEVICE} ({DEVICE_NAME})')
    print('Learning rate  :', MODEL_LEARNING_RATES[model_name])
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    with TRAINING_STRATEGY.scope():
        model = build_model(
            model_name,
            n_features=X_train.shape[1],
            learning_rate=MODEL_LEARNING_RATES[model_name],
            output_bias=OUTPUT_BIAS,
        )

    X_fit = X_train if model_name == 'MLP' else X_train_seq
    X_eval = X_test if model_name == 'MLP' else X_test_seq

    callbacks = [
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ]
    history = model.fit(
        X_fit,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        shuffle=False,
        callbacks=callbacks,
        verbose=1,
    )

    y_pred = model.predict(X_eval, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    y_pred = np.nan_to_num(y_pred, nan=np.nanmean(y_train), posinf=np.nanmean(y_train), neginf=0.0)
    metrics = compute_metrics(y_test, y_pred)
    repro_rows.append({'model': model_name, 'epochs_ran': len(history.history['loss']), **metrics})
    histories[model_name] = history.history
    print(metrics)

repro_df = pd.DataFrame(repro_rows)
repro_df

Training configuration
[INFO] Using GPU for deep learning training (Tesla T4).
Compute device : GPU
Device name    : Tesla T4
Batch size     : 1024 (auto-selected for GPU)
Output bias    : 6914.6699
Models to run  : ['Transformer', 'LSTM', 'MLP', 'RNN', 'CNN']

=== Transformer ===
Compute device : GPU (Tesla T4)
Learning rate  : 0.0001
Epoch 1/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 72s 60ms/step - loss: 9376357.0000 - mae: 2268.7402 - val_loss: 12012799.0000 - val_mae: 2475.5984 - learning_rate: 1.0000e-04
Epoch 2/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 9347395.0000 - mae: 2265.2507 - val_loss: 11972140.0000 - val_mae: 2471.2769 - learning_rate: 1.0000e-04
Epoch 3/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 9319048.0000 - mae: 2261.9504 - val_loss: 11928194.0000 - val_mae: 2466.6045 - learning_rate: 1.0000e-04
Epoch 4/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 9288265.0000 - mae: 2258.3445 - val_loss: 11873783.0000 - val_mae: 2460.4404 - learning_rate: 1

,model,epochs_ran,rmse,mse,r2,mae
0,Transformer,100,854.183089,7.296288e+05,0.922242,567.275085
1,LSTM,52,1133.515880,1.284858e+06,0.863070,778.707764
2,MLP,33,800.240628,6.403851e+05,0.931753,541.793091
3,RNN,65,1073.638149,1.152699e+06,0.877154,738.912903
4,CNN,33,795.314325,6.325249e+05,0.932590,516.395264


Training output aggregates Transformer, LSTM, MLP, RNN, and CNN performance in `repro_df`. `epochs_ran` reflects early stopping, while RMSE, MSE, R², and MAE are the primary comparison metrics against the paper.


## Reference Results

Compute RMSE, MSE, R², and MAE on the test set and compare reproduction results against `PAPER_REFERENCE`. Delta and percentage-difference columns show the gap between reproduction and paper values.


In [11]:
paper_df = pd.DataFrame(PAPER_REFERENCE).T.reset_index().rename(columns={'index': 'model'})
comparison = paper_df.merge(repro_df, on='model', suffixes=('_paper', '_reproduced'))

for metric in ['rmse', 'mse', 'r2', 'mae']:
    comparison[f'delta_{metric}'] = comparison[f'{metric}_reproduced'] - comparison[f'{metric}_paper']
    comparison[f'pct_diff_{metric}'] = np.where(
        comparison[f'{metric}_paper'] != 0,
        comparison[f'delta_{metric}'] / comparison[f'{metric}_paper'] * 100,
        np.nan,
    )

comparison[[
    'model',
    'rmse_paper', 'rmse_reproduced', 'delta_rmse', 'pct_diff_rmse',
    'mse_paper', 'mse_reproduced', 'delta_mse', 'pct_diff_mse',
    'r2_paper', 'r2_reproduced', 'delta_r2',
    'mae_paper', 'mae_reproduced', 'delta_mae', 'pct_diff_mae',
    'epochs_ran',
]]

,model,rmse_paper,rmse_reproduced,delta_rmse,pct_diff_rmse,mse_paper,mse_reproduced,delta_mse,pct_diff_mse,r2_paper,r2_reproduced,delta_r2,mae_paper,mae_reproduced,delta_mae,pct_diff_mae,epochs_ran
0,MLP,1322.57,806.081765,-516.488235,-39.051864,1750258.53,6.497678e+05,-1.100491e+06,-62.875895,0.89,0.930753,0.040753,835.54,544.012329,-291.527671,-34.890929,33
1,LSTM,1283.77,1138.790971,-144.979029,-11.293225,1646816.85,1.296845e+06,-3.499720e+05,-21.251421,0.91,0.861792,-0.048208,800.01,801.150085,1.140085,0.142509,46
2,CNN,1932.60,759.428362,-1173.171638,-60.704317,3734871.20,5.767314e+05,-3.158140e+06,-84.558197,0.74,0.938536,0.198536,1255.74,493.859589,-761.880411,-60.671828,34
3,RNN,1394.37,1080.975543,-313.394457,-22.475703,1945628.38,1.168508e+06,-7.771203e+05,-39.941865,0.88,0.875469,-0.004531,821.29,736.905640,-84.384360,-10.274612,61
4,Transformer,1214.00,877.469907,-336.530093,-27.720765,1475395.99,7.699534e+05,-7.054426e+05,-47.813777,0.96,0.917944,-0.042056,760.48,587.181702,-173.298298,-22.788015,100


The comparison table is the main reproduction summary: each metric shows paper value, reproduced value, absolute delta, and percentage difference. Smaller `delta_*` or `pct_diff_*` indicates closer alignment; large gaps should be read together with architecture/preprocessing deviations and PDF extraction inconsistencies.


## Best Baseline & Export (Reference)


In [12]:
best_reference = repro_df.sort_values(['rmse', 'mae']).iloc[0]
df_compare_ref = repro_df[['model', 'rmse', 'mse', 'r2', 'mae', 'epochs_ran']].copy()
df_compare_ref = df_compare_ref.rename(
    columns={
        'rmse': 'Reference RMSE',
        'mse': 'Reference MSE',
        'r2': 'Reference R2',
        'mae': 'Reference MAE',
        'epochs_ran': 'Epochs Ran',
    }
)
print('Best Baseline (Reference Preprocessing):')
display(best_reference)
display(df_compare_ref.sort_values('Reference RMSE'))


Best Baseline (Reference Preprocessing):


,4
model,CNN
epochs_ran,34
rmse,759.428362
mse,576731.4375
r2,0.938536
mae,493.859589


,model,Reference RMSE,Reference MSE,Reference R2,Reference MAE,Epochs Ran
4,CNN,759.428362,5.767314e+05,0.938536,493.859589,34
2,MLP,806.081765,6.497678e+05,0.930753,544.012329,33
0,Transformer,877.469907,7.699534e+05,0.917944,587.181702,100
3,RNN,1080.975543,1.168508e+06,0.875469,736.905640,61
1,LSTM,1138.790971,1.296845e+06,0.861792,801.150085,46


# Exploration Based on Our Preprocessing

This section follows the **`our_study_rosman.ipynb` data approach** (feature engineering + logarithmic `Sales` transform from *Using Logarithmic*), then trains the **same deep learning models** as the reference section (Transformer, LSTM, MLP, RNN, CNN).

Only the model family differs from `our_study_rosman` (DL instead of LR+XGBoost hybrid).


## Preprocessing

Pipeline aligned with `our_study_rosman.ipynb`:
- Merge `train.csv` + `store.csv`
- Create `IsPromo2Active` and `Promo2DurationWeeks`
- One-hot encode `StoreType`, `Assortment`, `StateHoliday`
- Sort columns (target `Sales` last)
- Drop competition columns and redundant promo metadata
- Keep `Date` for visualization only (excluded from model features)

Modeling preparation (next cells) follows the **Using Logarithmic** protocol:
- Keep rows with `Sales > 0`
- Target = `log1p(Sales)` (`Sales_log`)
- Chronological 80/20 split
- `MinMaxScaler` on features
- Report metrics on log scale and real scale (`expm1`)


In [7]:
# Load and merge Rossmann data (same base load as our_study_rosman)
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
store_df = pd.read_csv(STORE_PATH, low_memory=False)
data = pd.merge(train_df, store_df, how='left', on='Store')
data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
series = data.sort_values('Date').reset_index(drop=True)

# --- Feature engineering (our_study_rosman) ---
series['MonthStr'] = series['Date'].dt.strftime('%b')


def is_promo2_active(row):
    if row['Promo2'] == 1 and isinstance(row['PromoInterval'], str):
        return 1 if row['MonthStr'] in row['PromoInterval'].split(',') else 0
    return 0


series['IsPromo2Active'] = series.apply(is_promo2_active, axis=1)
series['Promo2DurationWeeks'] = np.where(
    series['Promo2'] == 1,
    (series['Date'].dt.year - series['Promo2SinceYear']) * 52
    + (series['Date'].dt.isocalendar().week - series['Promo2SinceWeek']),
    0,
)
series['Promo2DurationWeeks'] = series['Promo2DurationWeeks'].clip(lower=0).fillna(0)

cat_cols = ['StoreType', 'Assortment', 'StateHoliday']
series = pd.get_dummies(series, columns=cat_cols, drop_first=True)
boolean_cols = series.select_dtypes(include='bool').columns
series[boolean_cols] = series[boolean_cols].astype(int)

target_col = 'Sales'
feature_order = sorted([c for c in series.columns if c != target_col]) + [target_col]
series = series[feature_order]

series.drop(
    columns=[
        'CompetitionDistance',
        'CompetitionOpenSinceMonth',
        'CompetitionOpenSinceYear',
        'Promo2SinceWeek',
        'Promo2SinceYear',
        'PromoInterval',
        'MonthStr',
    ],
    inplace=True,
    errors='ignore',
)
series.dropna(inplace=True)

print('Our preprocessing complete.')
print('Shape:', series.shape)
print('Columns:', list(series.columns))
series.head()

Our preprocessing complete.
Shape: (1017209, 19)
Columns: ['Assortment_b', 'Assortment_c', 'Customers', 'Date', 'DayOfWeek', 'IsPromo2Active', 'Open', 'Promo', 'Promo2', 'Promo2DurationWeeks', 'SchoolHoliday', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'Store', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'Sales']


,Assortment_b,Assortment_c,Customers,Date,DayOfWeek,IsPromo2Active,Open,Promo,Promo2,Promo2DurationWeeks,SchoolHoliday,StateHoliday_a,StateHoliday_b,StateHoliday_c,Store,StoreType_b,StoreType_c,StoreType_d,Sales
0,0,1,0,2013-01-01,2,0,0,0,1,31.0,1,1,0,0,1115,0,0,1,0
1,0,0,0,2013-01-01,2,0,0,0,0,0.0,1,1,0,0,379,0,0,1,0
2,0,1,0,2013-01-01,2,0,0,0,0,0.0,1,1,0,0,378,0,0,0,0
3,0,1,0,2013-01-01,2,0,0,0,1,139.0,1,1,0,0,377,0,0,0,0
4,0,0,0,2013-01-01,2,0,0,0,0,0.0,1,1,0,0,376,0,0,0,0


### Modeling Data Preparation

Aligned with `our_study_rosman` **Using Logarithmic (RMSE, MSE, R2, MAE)**:
- Filter `Sales > 0`
- Target `Sales_log = log1p(Sales)` (raw `Sales` dropped from training columns)
- Chronological 80/20 split
- `MinMaxScaler` on input features
- Evaluation uses both log-scale metrics and real-scale metrics via `expm1`


In [8]:
# our_study_rosman — Using Logarithmic protocol (models remain Diamantini DL)
MAX_ROWS_OUR = None
TEST_SIZE_OUR = 0.20

data_to_transform = series[series['Sales'] > 0].copy()
data_to_transform['Sales_log'] = np.log1p(data_to_transform['Sales'])
data_to_train = data_to_transform.drop(columns=['Sales']).copy()

if MAX_ROWS_OUR is not None:
    data_to_train = data_to_train.tail(MAX_ROWS_OUR).copy()

n_train_our = int(len(data_to_train) * (1 - TEST_SIZE_OUR))
train_df_our = data_to_train.iloc[:n_train_our].copy()
test_df_our = data_to_train.iloc[n_train_our:].copy()

target_col_our = 'Sales_log'
feature_cols_our = [c for c in train_df_our.columns if c not in [target_col_our, 'Date']]

X_train_raw_our = train_df_our[feature_cols_our].to_numpy(dtype=np.float32)
y_train_our = train_df_our[target_col_our].to_numpy(dtype=np.float32)
X_test_raw_our = test_df_our[feature_cols_our].to_numpy(dtype=np.float32)
y_test_our = test_df_our[target_col_our].to_numpy(dtype=np.float32)
y_test_our_real = np.expm1(y_test_our).astype(np.float32)

scaler_our = MinMaxScaler()
X_train_our = scaler_our.fit_transform(X_train_raw_our).astype(np.float32)
X_test_our = scaler_our.transform(X_test_raw_our).astype(np.float32)

X_train_our = np.nan_to_num(X_train_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test_our = np.nan_to_num(X_test_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_train_our = np.nan_to_num(y_train_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_test_our = np.nan_to_num(y_test_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_test_our_real = np.nan_to_num(y_test_our_real, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

assert np.isfinite(X_train_our).all(), 'X_train_our contains NaN/Inf'
assert np.isfinite(X_test_our).all(), 'X_test_our contains NaN/Inf'
assert np.isfinite(y_train_our).all(), 'y_train_our contains NaN/Inf'
assert np.isfinite(y_test_our).all(), 'y_test_our contains NaN/Inf'

print('rows (Sales > 0):', len(data_to_train))
print('train:', len(train_df_our), 'test:', len(test_df_our), 'features:', len(feature_cols_our))
print('target: Sales_log = log1p(Sales)')
print('feature scaler: MinMaxScaler')
print('y_log range :', float(y_train_our.min()), float(y_train_our.max()))
print('y_real range:', float(np.expm1(y_train_our).min()), float(np.expm1(y_train_our).max()))
pd.Series(y_train_our).describe()


rows (Sales > 0): 844338
train: 675470 test: 168868 features: 17
target: Sales_log = log1p(Sales)
feature scaler: MinMaxScaler
y_log range : 3.8501474857330322 10.546340942382812
y_real range: 45.999996185302734 38037.0


,0
count,675470.000000
mean,8.750544
std,0.428035
min,3.850147
25%,8.479283
50%,8.752265
75%,9.026177
max,10.546341


## Modeling

Train the same deep learning models as the reference section (Transformer, LSTM, MLP, RNN, CNN) on the `our_study_rosman` logarithmic feature/target setup.

Models learn `Sales_log`; metrics are reported on **log scale** and **real scale** (`expm1`), matching the evaluation style in `our_study_rosman`.

#### Training


In [9]:
MODELS_TO_RUN_OUR = ['Transformer', 'LSTM', 'MLP', 'RNN', 'CNN']
EPOCHS_OUR = 100
BATCH_SIZE_OUR = 1024 if GPU_AVAILABLE else 256
VALIDATION_SPLIT_OUR = 0.10
PATIENCE_OUR = 12
OUTPUT_BIAS_OUR = float(np.mean(y_train_our))  # mean of Sales_log
MODEL_LEARNING_RATES_OUR = {
    'Transformer': 1e-4,
    'LSTM': 3e-4,
    'MLP': 5e-4,
    'RNN': 2e-4,
    'CNN': 5e-4,
}

X_train_seq_our = X_train_our[:, np.newaxis, :]
X_test_seq_our = X_test_our[:, np.newaxis, :]

print('=' * 60)
print('Our preprocessing (log) — training configuration')
print('=' * 60)
if GPU_AVAILABLE:
    print(f'[INFO] Using GPU for deep learning training ({DEVICE_NAME}).')
else:
    print('[WARN] GPU not available — falling back to CPU training.')
print(f'Compute device : {COMPUTE_DEVICE}')
print(f'Batch size     : {BATCH_SIZE_OUR} (auto-selected for {COMPUTE_DEVICE})')
print(f'Target         : Sales_log (log1p)')
print(f'Output bias    : {OUTPUT_BIAS_OUR:.4f}')
print(f'Models to run  : {MODELS_TO_RUN_OUR}')
print('=' * 60)

repro_rows_our = []
histories_our = {}

for model_name in MODELS_TO_RUN_OUR:
    print(f'\n=== {model_name} (Our Preprocessing / log) ===')
    print(f'Compute device : {COMPUTE_DEVICE} ({DEVICE_NAME})')
    print('Learning rate  :', MODEL_LEARNING_RATES_OUR[model_name])
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    with TRAINING_STRATEGY.scope():
        model = build_model(
            model_name,
            n_features=X_train_our.shape[1],
            learning_rate=MODEL_LEARNING_RATES_OUR[model_name],
            output_bias=OUTPUT_BIAS_OUR,
        )

    X_fit = X_train_our if model_name == 'MLP' else X_train_seq_our
    X_eval = X_test_our if model_name == 'MLP' else X_test_seq_our

    callbacks = [
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE_OUR, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ]
    history = model.fit(
        X_fit,
        y_train_our,
        epochs=EPOCHS_OUR,
        batch_size=BATCH_SIZE_OUR,
        validation_split=VALIDATION_SPLIT_OUR,
        shuffle=False,
        callbacks=callbacks,
        verbose=1,
    )

    y_pred_log = model.predict(X_eval, batch_size=BATCH_SIZE_OUR, verbose=0).reshape(-1)
    y_pred_log = np.nan_to_num(
        y_pred_log,
        nan=np.nanmean(y_train_our),
        posinf=np.nanmean(y_train_our),
        neginf=0.0,
    )
    y_pred_real = np.expm1(y_pred_log).astype(np.float32)
    y_pred_real = np.nan_to_num(y_pred_real, nan=0.0, posinf=0.0, neginf=0.0)

    metrics_log = compute_metrics(y_test_our, y_pred_log)
    metrics_real = compute_metrics(y_test_our_real, y_pred_real)

    repro_rows_our.append({
        'model': model_name,
        'epochs_ran': len(history.history['loss']),
        'rmse_log': metrics_log['rmse'],
        'mse_log': metrics_log['mse'],
        'r2_log': metrics_log['r2'],
        'mae_log': metrics_log['mae'],
        'rmse': metrics_real['rmse'],
        'mse': metrics_real['mse'],
        'r2': metrics_real['r2'],
        'mae': metrics_real['mae'],
    })
    histories_our[model_name] = history.history
    print('log-scale :', metrics_log)
    print('real-scale:', metrics_real)

repro_df_our = pd.DataFrame(repro_rows_our)
repro_df_our


Our preprocessing (log) — training configuration
[INFO] Using GPU for deep learning training (Tesla T4).
Compute device : GPU
Batch size     : 1024 (auto-selected for GPU)
Target         : Sales_log (log1p)
Output bias    : 8.7505
Models to run  : ['Transformer', 'LSTM', 'MLP', 'RNN', 'CNN']

=== Transformer (Our Preprocessing / log) ===
Compute device : GPU (Tesla T4)
Learning rate  : 0.0001
Epoch 1/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 65s 53ms/step - loss: 0.3886 - mae: 0.4639 - val_loss: 0.0397 - val_mae: 0.1577 - learning_rate: 1.0000e-04
Epoch 2/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 0.1784 - mae: 0.3227 - val_loss: 0.0372 - val_mae: 0.1525 - learning_rate: 1.0000e-04
Epoch 3/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - loss: 0.1307 - mae: 0.2792 - val_loss: 0.0338 - val_mae: 0.1456 - learning_rate: 1.0000e-04
Epoch 4/100
594/594 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 0.1027 - mae: 0.2509 - val_loss: 0.0301 - val_mae: 0.1376 - learning_rate: 1.0000e-04
Epoch 5/

,model,epochs_ran,rmse_log,mse_log,r2_log,mae_log,rmse,mse,r2,mae
0,Transformer,100,0.145684,0.021224,0.875067,0.112412,1087.156900,1181910.125,0.874059,783.097046
1,LSTM,23,0.168792,0.028491,0.832292,0.133934,1356.818291,1840955.875,0.803833,955.949219
2,MLP,51,0.152721,0.023324,0.862706,0.118388,1132.480739,1282512.625,0.863339,815.735535
3,RNN,19,0.198841,0.039538,0.767262,0.159405,1626.420764,2645244.500,0.718131,1142.609985
4,CNN,39,0.164253,0.026979,0.841188,0.128437,1211.943738,1468807.625,0.843488,885.545898


## Our Results

Deep learning models trained with the `our_study_rosman` **Using Logarithmic** data protocol.

- `*_log` columns: metrics on `Sales_log`
- `rmse` / `mse` / `r2` / `mae`: metrics on real `Sales` after `expm1`
- Ranking uses **real-scale RMSE** (comparable to Reference section and paper units)


## Best Baseline & Export (Our Preprocessing)

Best model is selected by real-scale RMSE (then MAE), consistent with Reference ranking and `our_study_rosman` real-scale reporting.


In [ ]:
best_our = repro_df_our.sort_values(['rmse', 'mae']).iloc[0]
df_compare_our = repro_df_our[
    ['model', 'rmse', 'mse', 'r2', 'mae', 'rmse_log', 'mse_log', 'r2_log', 'mae_log', 'epochs_ran']
].copy()
df_compare_our = df_compare_our.rename(
    columns={
        'rmse': 'Our Real RMSE',
        'mse': 'Our Real MSE',
        'r2': 'Our Real R2',
        'mae': 'Our Real MAE',
        'rmse_log': 'Our Log RMSE',
        'mse_log': 'Our Log MSE',
        'r2_log': 'Our Log R2',
        'mae_log': 'Our Log MAE',
        'epochs_ran': 'Epochs Ran',
    }
)

print('Best Baseline (Our Preprocessing / log -> real scale):')
display(best_our)
display(df_compare_our.sort_values('Our Real RMSE'))


# Summary

## Experimental Setup

Two approaches compared using 5 deep learning models on Rossmann daily sales:

| Aspect | Reference (Paper-style) | Our Preprocessing (`our_study_rosman`) |
|--------|-------------------------|----------------------------------------|
| Split | Chronological 80/20 | Chronological 80/20 |
| Row filter | `Open == 1` | `Sales > 0` (Using Logarithmic) |
| Target | Raw `Sales` | `Sales_log = log1p(Sales)` |
| Eval scale | Real `Sales` | Log metrics + real metrics via `expm1` |
| Features | Promo2 activity/duration, competition fill-zero, one-hot categoricals | Promo2 activity/duration, one-hot categoricals, drop competition + redundant promo metadata |
| Scaling | `StandardScaler` on features | `MinMaxScaler` on features |
| Models | Transformer, LSTM, MLP, RNN, CNN | Same DL models (not LR+XGBoost) |

## Models Evaluated

| # | Model | Type |
|---|-------|------|
| 1 | Transformer | Deep Learning |
| 2 | LSTM | Deep Learning |
| 3 | MLP | Deep Learning |
| 4 | RNN | Deep Learning |
| 5 | CNN | Deep Learning |

## Key Metrics

- Reference: RMSE, MSE, R2, MAE on raw `Sales`; also compared to verified `PAPER_REFERENCE`.
- Our: same metrics on **log scale** and **real scale** (after `expm1`), following `our_study_rosman` reporting.
- Cross-section ranking uses **real-scale RMSE**.

## Key Findings

- **Reference best (raw Sales)**: CNN was strongest in the saved Reference run (real RMSE ~759).
- **Our section protocol**: Now matches `our_study_rosman` Using Logarithmic (positive sales, `log1p` target, `MinMaxScaler`), while keeping Diamantini DL architectures.
- **Re-run required**: Saved Our-section numeric results above were from the previous raw-Sales setup; re-run the Our Modeling cells to refresh `repro_df_our` / best model under the log protocol.
- **Comparability note**: Reference and Our now differ in both feature pipeline and target transform; interpret gaps as combined preprocessing+target effects, not feature engineering alone.

## Limitations

- **Not a full `our_study` replica**: Models remain DL baselines; LR+XGBoost residual hybrid is intentionally excluded.
- **Paper ambiguity**: Diamantini hyperparameters are not fully specified; architectures here are documented equivalents.
- **Setup mismatch vs paper**: Different filters/splits/transforms mean absolute paper deltas should be read cautiously.
- **Compute sensitivity**: GPU/CPU, TensorFlow version, batch size, and initialization can change results even with a fixed seed.
